In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "experiments"))

from evaluation.metrics import (
    metric_sums_for_mask,
    finalise_metric_rows,
    crps_per_element,
)

torch.set_default_dtype(torch.float32)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 123
M = 64
B = 512
NT = 16
DY = 1

torch.manual_seed(SEED)

xt = torch.linspace(-4.0, 4.0, NT, device=DEVICE).view(1, NT, 1).repeat(B, 1, 1)

print("device:", DEVICE)
print("xt:", xt.shape)

device: cpu
xt: torch.Size([512, 16, 1])


In [6]:
def summarise_case(name, samples, target, alpha=1.0):
    """Return one-row final metrics dataframe for synthetic toy samples."""
    row = metric_sums_for_mask(
        samples=samples,
        target=target,
        mask=None,
        alpha=alpha,
    )

    row.update(
        {
            "model_name": name,
            "checkpoint_path": "metric_sanity",
            "eval_set": "toy",
            "region": "all",
            "context_bucket": "all",
        }
    )

    return finalise_metric_rows([row])


def show_cases(case_dict):
    frames = []
    for name, (samples, target) in case_dict.items():
        frames.append(summarise_case(name, samples, target))

    out = pd.concat(frames, ignore_index=True)

    cols = [
        "model_name",
        "rmse_pooled",
        "crps",
        "energy_score",
        "spread_skill_ratio",
        "coverage_50",
        "coverage_80",
        "coverage_90",
        "coverage_95",
        "width_90",
    ]

    return out[cols].sort_values("model_name").reset_index(drop=True)


def normal_samples(mean, std, shape):
    return mean + std * torch.randn(shape, device=DEVICE)


def student_t_samples(df, shape):
    dist = torch.distributions.StudentT(df=torch.tensor(float(df), device=DEVICE))
    return dist.sample(shape)


def bimodal_samples(shape, loc=2.0, scale=0.3, shared_mode_per_task=False):
    """Two-component Gaussian mixture.

    If shared_mode_per_task=True:
        one mode sign is shared across all target locations within each task.
    Else:
        each target element chooses its mode independently.
    """
    if shared_mode_per_task:
        if len(shape) == 3:
            signs = torch.where(
                torch.rand(shape[0], 1, 1, device=DEVICE) < 0.5,
                -1.0,
                1.0,
            )
        elif len(shape) == 4:
            signs = torch.where(
                torch.rand(shape[0], shape[1], 1, 1, device=DEVICE) < 0.5,
                -1.0,
                1.0,
            )
        else:
            raise ValueError(f"Unsupported shape: {shape}")
    else:
        signs = torch.where(torch.rand(shape, device=DEVICE) < 0.5, -1.0, 1.0)

    return loc * signs + scale * torch.randn(shape, device=DEVICE)

## Marginal metric sanity checks

In [7]:
torch.manual_seed(SEED)

target_normal = normal_samples(0.0, 1.0, (B, NT, DY))

cases = {
    "normal_calibrated": (
        normal_samples(0.0, 1.0, (M, B, NT, DY)),
        target_normal,
    ),
    "normal_underdispersed": (
        normal_samples(0.0, 0.5, (M, B, NT, DY)),
        target_normal,
    ),
    "normal_overdispersed": (
        normal_samples(0.0, 2.0, (M, B, NT, DY)),
        target_normal,
    ),
    "normal_biased": (
        normal_samples(0.5, 1.0, (M, B, NT, DY)),
        target_normal,
    ),
}

normal_results = show_cases(cases)
normal_results

,model_name,rmse_pooled,crps,energy_score,spread_skill_ratio,coverage_50,coverage_80,coverage_90,coverage_95,width_90
0,normal_biased,1.133944,0.638443,3.148687,0.889351,0.440063,0.722168,0.821167,0.883423,3.141645
1,normal_calibrated,1.013717,0.567109,2.800931,0.995116,0.491577,0.774902,0.870483,0.922363,3.145944
2,normal_overdispersed,1.037388,0.657222,3.248201,1.942876,0.782959,0.978027,0.995728,0.999146,6.286964
3,normal_underdispersed,1.006485,0.610940,3.026283,0.501316,0.261963,0.466064,0.569092,0.637451,1.572699


## Heavy-tail sanity checks

In [8]:
torch.manual_seed(SEED)

df = 3.0
target_t = student_t_samples(df, (B, NT, DY))

# Student-t variance is df / (df - 2) for df > 2.
t_std = (df / (df - 2.0)) ** 0.5

cases = {
    "student_t_calibrated": (
        student_t_samples(df, (M, B, NT, DY)),
        target_t,
    ),
    "student_t_gaussian_matched_variance": (
        normal_samples(0.0, t_std, (M, B, NT, DY)),
        target_t,
    ),
    "student_t_gaussian_underdispersed": (
        normal_samples(0.0, 1.0, (M, B, NT, DY)),
        target_t,
    ),
}

t_results = show_cases(cases)
t_results

,model_name,rmse_pooled,crps,energy_score,spread_skill_ratio,coverage_50,coverage_80,coverage_90,coverage_95,width_90
0,student_t_calibrated,1.689978,0.824148,4.473219,1.025364,0.493408,0.778564,0.869873,0.923584,4.504758
1,student_t_gaussian_matched_variance,1.690795,0.846580,4.500040,1.031630,0.651733,0.866699,0.919189,0.945923,5.434143
2,student_t_gaussian_underdispersed,1.681409,0.834188,4.617600,0.598960,0.437744,0.692749,0.777222,0.829346,3.140592


## Bimodal marginal sanity checks

In [9]:
torch.manual_seed(SEED)

target_bimodal = bimodal_samples((B, NT, DY), shared_mode_per_task=False)

cases = {
    "bimodal_calibrated": (
        bimodal_samples((M, B, NT, DY), shared_mode_per_task=False),
        target_bimodal,
    ),
    "bimodal_mode_collapsed_left": (
        -2.0 + 0.3 * torch.randn(M, B, NT, DY, device=DEVICE),
        target_bimodal,
    ),
    "bimodal_mean_collapsed": (
        0.0 + 0.3 * torch.randn(M, B, NT, DY, device=DEVICE),
        target_bimodal,
    ),
}

bimodal_results = show_cases(cases)
bimodal_results

,model_name,rmse_pooled,crps,energy_score,spread_skill_ratio,coverage_50,coverage_80,coverage_90,coverage_95,width_90
0,bimodal_calibrated,2.036092,1.084482,5.664088,1.000684,0.490967,0.783325,0.877686,0.925293,4.712693
1,bimodal_mean_collapsed,2.018315,1.826319,7.314272,0.149876,0.000000,0.000000,0.000000,0.000000,0.942504
2,bimodal_mode_collapsed_left,2.838544,1.994273,10.483474,0.106539,0.234619,0.389648,0.437622,0.463623,0.943659


## Function-level coherence sanity check

In [10]:
torch.manual_seed(SEED)

target_shared_mode = bimodal_samples(
    (B, NT, DY),
    loc=2.0,
    scale=0.15,
    shared_mode_per_task=True,
)

samples_shared_mode = bimodal_samples(
    (M, B, NT, DY),
    loc=2.0,
    scale=0.15,
    shared_mode_per_task=True,
)

samples_independent_modes = bimodal_samples(
    (M, B, NT, DY),
    loc=2.0,
    scale=0.15,
    shared_mode_per_task=False,
)

cases = {
    "joint_correct_shared_mode": (
        samples_shared_mode,
        target_shared_mode,
    ),
    "joint_wrong_independent_modes": (
        samples_independent_modes,
        target_shared_mode,
    ),
}

joint_results = show_cases(cases)
joint_results

,model_name,rmse_pooled,crps,energy_score,spread_skill_ratio,coverage_50,coverage_80,coverage_90,coverage_95,width_90
0,joint_correct_shared_mode,2.027465,1.047466,4.235020,0.997103,0.478638,0.768433,0.863525,0.915894,4.356230
1,joint_wrong_independent_modes,2.024261,1.044634,5.632268,0.998563,0.476318,0.771484,0.872192,0.917114,4.356276


## Fair CRPS sanity check

In [11]:
torch.manual_seed(SEED)

target = normal_samples(0.0, 1.0, (B, NT, DY))

samples_64 = normal_samples(0.0, 1.0, (64, B, NT, DY))
samples_512 = normal_samples(0.0, 1.0, (512, B, NT, DY))

crps_64_fair = crps_per_element(samples_64, target, alpha=1.0).mean().item()
crps_64_ordinary = crps_per_element(samples_64, target, alpha=0.0).mean().item()
crps_512_fair = crps_per_element(samples_512, target, alpha=1.0).mean().item()

print("CRPS M=64 fair:     ", crps_64_fair)
print("CRPS M=64 ordinary: ", crps_64_ordinary)
print("CRPS M=512 fair:    ", crps_512_fair)
print("ordinary - fair:    ", crps_64_ordinary - crps_64_fair)
print("fair64 - fair512:   ", crps_64_fair - crps_512_fair)

CRPS M=64 fair:      0.5671086311340332
CRPS M=64 ordinary:  0.575931966304779
CRPS M=512 fair:     0.5655838251113892
ordinary - fair:     0.00882333517074585
fair64 - fair512:    0.001524806022644043


## Assertions

In [12]:
def get_row(df, name):
    return df[df["model_name"] == name].iloc[0]


normal_cal = get_row(normal_results, "normal_calibrated")
normal_under = get_row(normal_results, "normal_underdispersed")
normal_over = get_row(normal_results, "normal_overdispersed")
normal_biased = get_row(normal_results, "normal_biased")

assert 0.87 <= normal_cal["coverage_90"] <= 0.93
assert normal_under["coverage_90"] < normal_cal["coverage_90"]
assert normal_over["width_90"] > normal_cal["width_90"]
assert normal_biased["crps"] > normal_cal["crps"]

joint_correct = get_row(joint_results, "joint_correct_shared_mode")
joint_wrong = get_row(joint_results, "joint_wrong_independent_modes")

assert joint_wrong["energy_score"] > joint_correct["energy_score"]

assert crps_64_ordinary > crps_64_fair

print("All metric sanity checks passed.")

All metric sanity checks passed.
